In [ ]:
import pandas as pd

df = pd.read_csv("/home/hvgupta/COMP4222/COMP4222_F25-26/src/one_year_company_info.csv")

In [ ]:
df.isnull().sum()

In [ ]:
len(df["Symbol"].unique())

In [ ]:
df.columns

In [ ]:
df["Weight"]

The current data in this file is outdated, I have mostly gotten it since it has the ticker for different stocks.

Some of the columns here are interesting and can be used to in our analysis

I think we can focus on using the following metrics for edges/nodes?(other than the tickers or the identifiers):
  - Market Cap: could show the much volatility there could be 
  - Weight: %maket cap of the company to the total market cap of S&P500


(for now I am gathering data for 500 companies, but we can choose a subset if we want)

we would definetly need to add more

In [ ]:
import pandas as pd
import yfinance as yf
import ta



In [ ]:
aapl = yf.download("AAPL", start="2023-01-01", end="2025-11-01")
msft = yf.download("MSFT", start="2023-01-01", end="2025-11-01")

In [ ]:
h = pd.merge(aapl, msft, left_index=True, right_index=True, suffixes=("_AAPL", "_MSFT"))

In [ ]:
h

In [ ]:
h.corr()

In [ ]:
l = h.copy()

In [ ]:
# Add a new column to the entire dataframe for the filtered date range
l.loc['2023-01-01':'2023-03-31', ('new_column', 'AAPL')] = 'your_value'

In [ ]:
l

In [ ]:
h.loc[:,"Close"]

In [ ]:
with open("test.csv", "w") as f:
    f.write(h.to_csv())

In [ ]:
# Example: download 2 years of data for one stock
data = yf.download("AAPL", start="2023-01-01", end="2025-01-01")

# --- Derived features ---
data["return_1d"] = data["Close"].pct_change()
data["return_5d"] = data["Close"].pct_change(5)
data["volatility_20d"] = data["Close"].pct_change().rolling(20).std()

# Trend features
data["ma_5"] = data["Close"].rolling(5).mean()
data["ma_20"] = data["Close"].rolling(20).mean()
data["ma_ratio"] = data["ma_5"] / data["ma_20"]

# RSI and MACD (from ta library)
data["rsi"] = ta.momentum.RSIIndicator(data["Close"], window=14).rsi()
macd = ta.trend.MACD(data["Close"])
data["macd"] = macd.macd()
data["macd_signal"] = macd.macd_signal()

# Volume-related
data["obv"] = ta.volume.OnBalanceVolumeIndicator(data["Close"], data["Volume"]).on_balance_volume()
data["vol_zscore"] = (data["Volume"] - data["Volume"].rolling(20).mean()) / data["Volume"].rolling(20).std()

# Price range ratio
data["price_range"] = (data["High"] - data["Low"]) / data["Close"]

In [ ]:
data

In [ ]:
import pandas as pd

df = pd.read_csv("./src/one_year_company_info.csv")

In [ ]:
AAPL_EODs = df[df["Symbol"] == "AAPL"]

In [ ]:
AAPL_EODs

In [ ]:
import talib

ROCP_5 = talib.ROCP(AAPL_EODs["Close"].to_numpy(), timeperiod=5)
ROCP_20 = talib.ROCP(AAPL_EODs["Close"].to_numpy(), timeperiod=20)
NATR_5 = talib.NATR(AAPL_EODs["High"].to_numpy(), AAPL_EODs["Low"].to_numpy(), AAPL_EODs["Close"].to_numpy(), timeperiod=5)

In [ ]:
NATR_5

In [ ]:
talib.MOM(AAPL_EODs["Close"].to_numpy(), timeperiod=5)
talib.MOM

In [ ]:
talib.ADXR(AAPL_EODs["High"].to_numpy(), AAPL_EODs["Low"].to_numpy(), AAPL_EODs["Close"].to_numpy(), timeperiod=5)

In [ ]:
talib.BETA()

In [ ]:
import pandas as pd
import requests
from io import StringIO

# Add headers to avoid 403 Forbidden error
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(url, headers=headers)
sp500 = pd.read_html(StringIO(response.text))[0]
tickers = sp500['Symbol'].tolist()


In [ ]:
sp500

In [ ]:
crta_stock_data = yf.download("CTRA", start="2020-01-01", end="2025-11-01")

In [ ]:
sp500[sp500["Founded"].str.match(r"\d{4} \(.+\)")]

In [ ]:
ctra = yf.Ticker("CTRA")


In [ ]:
q_income = ctra.quarterly_income_stmt
q_balance = ctra.quarterly_balance_sheet
q_earnings = ctra.quarterly_balance_sheet

In [ ]:
prices = crta_stock_data.loc["2020-01-01":"2021-01-01", "Close"]

In [ ]:
ctra.get_history_metadata()

In [ ]:
import yfinance as yf

ticker = yf.Ticker("CTRA")
q_earnings = ticker.get_earnings_dates()
q_earnings.sort_index()


In [1]:
import requests
url = "https://www.sec.gov/files/company_tickers.json"
headers = {'User-Agent': 'YourAppName/1.0 (hvgupta@outlook.in)'}  # Replace with your details
response = requests.get(url, headers=headers)
if response.status_code != 200:
    raise Exception(f"Failed to fetch data: {response.status_code}")


In [2]:
ticker_to_cik_map = {info['ticker']: str(info['cik_str']).zfill(10) for info in response.json().values()}


In [3]:
import requests
def get_sec_facts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    headers = {'User-Agent': 'YourAppName/1.0 (your.email@example.com)'}  # Replace with your details
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to fetch data: {response.status_code}")

In [4]:
facts = get_sec_facts(ticker_to_cik_map["CTRA"])

In [5]:
import pandas as pd

def extract_quarterly_data(facts: dict, metric_name: str, unit: str) -> pd.DataFrame:
       
    if "us-gaap" not in facts:
        return pd.DataFrame()
    
    if metric_name not in facts["us-gaap"]:
        return pd.DataFrame()
    
    if unit not in facts["us-gaap"][metric_name]["units"]:
        return pd.DataFrame()
    
    data = facts["us-gaap"][metric_name]["units"][unit]
    df = pd.DataFrame(data)
    df['end'] = pd.to_datetime(df['end'])
    df = df.sort_values(by='end').reset_index(drop=True)
    
    return df

In [6]:
from src.gather_company_info import get_PE_ratio_data
from src.market_data_fetcher import *

In [10]:
crta_hist_data = get_ticker_historical_prices("CTRA", "2020-01-01", "2025-01-01")

2025-11-11 09:59:57,349 - src.logger - INFO - Fetching historical prices for CTRA from 2020-01-01 to 2025-01-01


/home/hvgupta/COMP4222/COMP4222_F25-26/src/market_data_fetcher/financial_api_functions.py:52: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

2025-11-11 09:59:58,053 - src.logger - INFO - Successfully fetched historical prices for CTRA


In [8]:
PE_info = get_PE_ratio_data("CTRA", crta_hist_data, facts["facts"], 2020, 2025)

2025-11-11 01:19:00,658 - src.logger - INFO - Extracting quarterly data for EarningsPerShareBasic in USD/shares
2025-11-11 01:19:00,752 - src.logger - INFO - Successfully extracted quarterly data for EarningsPerShareBasic
2025-11-11 01:19:00,756 - src.logger - INFO - Starting to clean EPS table with 223 rows
2025-11-11 01:19:00,758 - src.logger - DEBUG - Skipping row at index 0 with end year 2007 outside range 2020-2025
2025-11-11 01:19:00,759 - src.logger - DEBUG - Skipping row at index 1 with end year 2008 outside range 2020-2025
2025-11-11 01:19:00,760 - src.logger - DEBUG - Skipping row at index 2 with end year 2008 outside range 2020-2025
2025-11-11 01:19:00,761 - src.logger - DEBUG - Skipping row at index 3 with end year 2008 outside range 2020-2025
2025-11-11 01:19:00,761 - src.logger - DEBUG - Skipping row at index 4 with end year 2008 outside range 2020-2025
2025-11-11 01:19:00,762 - src.logger - DEBUG - Skipping row at index 5 with end year 2008 outside range 2020-2025
2025-1

/home/hvgupta/COMP4222/COMP4222_F25-26/src/helper_functions.py:123: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  filtered_eps = pd.concat(


In [14]:
test = extract_quarterly_data(facts["facts"], 'EarningsPerShareBasic', 'USD/shares')

2025-11-11 10:01:35,747 - src.logger - INFO - Extracting quarterly data for EarningsPerShareBasic in USD/shares
2025-11-11 10:01:35,759 - src.logger - INFO - Successfully extracted quarterly data for EarningsPerShareBasic


In [15]:
test

,start,end,val,accn,fy,fp,form,filed,frame
0,2007-01-01,2007-12-31,1.73,0001193125-10-042257,2009,FY,10-K,2010-02-26,CY2007
1,2008-01-01,2008-06-30,1.03,0001193125-09-159490,2009,Q2,10-Q,2009-07-30,NaN
2,2008-04-01,2008-06-30,0.55,0001193125-09-159490,2009,Q2,10-Q,2009-07-30,CY2008Q2
3,2008-01-01,2008-09-30,1.68,0001193125-09-217450,2009,Q3,10-Q,2009-10-29,NaN
4,2008-07-01,2008-09-30,0.65,0001193125-09-217450,2009,Q3,10-Q,2009-10-29,CY2008Q3
...,...,...,...,...,...,...,...,...,...
218,2025-01-01,2025-03-31,0.68,0000858470-25-000111,2025,Q1,10-Q,2025-05-06,CY2025Q1
219,2025-01-01,2025-06-30,1.35,0000858470-25-000165,2025,Q2,10-Q,2025-08-05,NaN
220,2025-04-01,2025-06-30,0.67,0000858470-25-000165,2025,Q2,10-Q,2025-08-05,CY2025Q2
221,2025-01-01,2025-09-30,1.77,0000858470-25-000199,2025,Q3,10-Q,2025-11-04,NaN


In [9]:
PE_info

,start,end,eps,fp,price,PE_ratio
0,2020-01-01,2020-03-31,0.14,Q1,13.165380,94.038432
1,2020-04-01,2020-06-30,0.08,Q2,13.227336,165.341699
2,2020-01-01,2020-09-30,0.17,Q3,13.434016,79.023625
3,2020-01-01,2020-12-31,0.11,Q4,12.674378,115.221622
4,2020-01-01,2020-12-31,0.50,FY,12.674378,25.348757
5,2021-01-01,2021-03-31,0.32,Q1,14.699677,45.936492
6,2021-04-01,2021-06-30,0.08,Q2,13.753370,171.917129
7,2021-07-01,2021-09-30,0.16,Q3,17.256395,107.852471
8,2021-01-01,2021-12-31,1.74,Q4,15.652453,8.995663
9,2021-01-01,2021-12-31,2.30,FY,15.652453,6.805415


In [ ]:
crta_hist_data.reindex(eps_table["end"])

Price,Close,High,Low,Open,Volume
Ticker,CTRA,CTRA,CTRA,CTRA,CTRA
Date,,,,,
2020-01-02,13.110493,13.392030,12.882220,13.331158,8088300
2020-01-03,13.148536,13.346373,13.042007,13.323545,6896300
2020-01-06,13.255064,13.300718,13.034399,13.270283,8388500
2020-01-07,13.331154,13.338763,12.981134,13.133317,6983600
2020-01-08,12.821345,13.224627,12.707209,13.125709,10755300
...,...,...,...,...,...
2024-12-24,24.093304,24.103046,23.343128,23.567207,3660500
2024-12-26,23.947165,23.986136,23.664632,23.947165,3686200


In [13]:
pd.Timestamp("2022-12-31").weekday()

5

In [33]:
eps_table = extract_quarterly_data(facts["facts"], 'EarningsPerShareBasic', 'USD/shares')

2025-11-11 10:48:33,361 - src.logger - INFO - Extracting quarterly data for EarningsPerShareBasic in USD/shares
2025-11-11 10:48:33,391 - src.logger - INFO - Successfully extracted quarterly data for EarningsPerShareBasic


In [34]:
eps_table["start"] = pd.to_datetime(eps_table["start"], errors="coerce")
eps_table["end"] = pd.to_datetime(eps_table["end"], errors="coerce")

In [12]:
eps_table = eps_table.dropna(subset=["start", "end", "val"])

In [14]:
filtered_eps = pd.DataFrame(columns=["start","end", "eps", "fp"])

In [79]:
from pandas import Timestamp
def determine_quarter(end_date: Timestamp):
    return end_date.quarter

def get_start_and_end_of_quarter(year: int, quarter: int):
    if quarter == 1:
        return Timestamp(year=year, month=1, day=1), Timestamp(year=year, month=3, day=31)
    elif quarter == 2:
        return Timestamp(year=year, month=4, day=1), Timestamp(year=year, month=6, day=30)
    elif quarter == 3:
        return Timestamp(year=year, month=7, day=1), Timestamp(year=year, month=9, day=30)
    elif quarter == 4:
        return Timestamp(year=year, month=10, day=1), Timestamp(year=year, month=12, day=31)
    elif quarter == 0:
        return Timestamp(year=year, month=1, day=1), Timestamp(year=year, month=12, day=31)
    else:
        raise ValueError("Quarter must be between 1 and 4")

In [ ]:
def _get_sum_of_prev_quarters(y_q_to_eps_map: dict, year: int):
    sum_prev = (
        y_q_to_eps_map.get((year, "Q1"), {}).get("eps", 0)
        + y_q_to_eps_map.get((year, "Q2"), {}).get("eps", 0)
        + y_q_to_eps_map.get((year, "Q3"), {}).get("eps", 0)
    )
    logger.debug(f"Sum of previous quarters for year {year}: {sum_prev}")
    return sum_prev


def _eps_get_start_end_filter(eps_table: pd.DataFrame, year:int, quarter: str):
    start, end = get_start_and_end_of_quarter(year, int(quarter[1]) if quarter[1].isdigit() else 0)
    return (eps_table["start"] >= start) & (eps_table["end"] <= end)


def clean_eps_table(eps_table: pd.DataFrame, start_year: int, end_year: int):
    logger.info(f"Starting to clean EPS table with {len(eps_table)} rows")
    filtered_eps = pd.DataFrame(columns=["start", "end", "eps", "fp"])
    y_q_to_eps_map = {}
    for year in range(start_year, end_year + 1):
        for quarter in ["Q1","Q2","Q3","Q4", "FY"]:
            subset = eps_table[
                (eps_table["end"].dt.year == year)
                & (eps_table["fp"] == quarter)
                & _eps_get_start_end_filter(eps_table, year, quarter)
            ].sort_values("filed", ascending=False)

            if subset.empty:
                logger.debug(f"No data for year {year} quarter {quarter}")
                continue

            y_q_eps_table = subset.iloc[0]  # safe: subset has at least one row
            y_q_to_eps_map[(year, quarter)] = {
                "start": y_q_eps_table["start"],
                "end": y_q_eps_table["end"],
                "eps": y_q_eps_table["val"],
                "fp": quarter
            }

        # Q4 fallback: compute from FY if Q4 missing (do this per-year)
        if (year, "Q4") not in y_q_to_eps_map and (year, "FY") in y_q_to_eps_map:
            start, end = get_start_and_end_of_quarter(year, 4)
            y_q_to_eps_map[(year, "Q4")] = {
                "start": start,
                "end": end,
                "eps": y_q_to_eps_map[(year, "FY")]["eps"] - _get_sum_of_prev_quarters(y_q_to_eps_map, year),
                "fp": "Q4"
            }

    filtered_eps = pd.concat(
        [filtered_eps, pd.DataFrame(list(y_q_to_eps_map.values()))], ignore_index=True
    )
    logger.info(f"Cleaned EPS table: {len(filtered_eps)} rows after processing")
    return filtered_eps

filtered_eps = clean_eps_table(eps_table, 2020, 2025)

2025-11-11 12:17:59,548 - src.logger - INFO - Starting to clean EPS table with 223 rows
2025-11-11 12:17:59,602 - src.logger - DEBUG - No data for year 2020 quarter Q4


ValueError: invalid literal for int() with base 10: 'Y'

2025-11-11 12:15:34,700 - src.logger - INFO - Starting to clean EPS table with 223 rows


IndexError: single positional indexer is out-of-bounds

In [ ]:
eps_table[eps_table["end"].dt.year == 2020 ].sort_values(["fp","filed"], ascending=[True, False])

,start,end,val,accn,fy,fp,form,filed,frame
166,2020-01-01,2020-12-31,0.50,0000858470-23-000011,2022,FY,10-K,2023-02-27,CY2020
167,2020-01-01,2020-12-31,0.50,0000858470-22-000009,2021,FY,10-K,2022-03-01,NaN
153,2020-01-01,2020-03-31,0.14,0000858470-21-000013,2020,FY,10-K,2021-02-26,NaN
158,2020-04-01,2020-06-30,0.08,0000858470-21-000013,2020,FY,10-K,2021-02-26,NaN
160,2020-07-01,2020-09-30,-0.04,0000858470-21-000013,2020,FY,10-K,2021-02-26,NaN
165,2020-10-01,2020-12-31,0.33,0000858470-21-000013,2020,FY,10-K,2021-02-26,CY2020Q4
168,2020-01-01,2020-12-31,0.50,0000858470-21-000013,2020,FY,10-K,2021-02-26,NaN
152,2020-01-01,2020-03-31,0.14,0000858470-21-000022,2021,Q1,10-Q,2021-04-30,CY2020Q1
154,2020-01-01,2020-03-31,0.14,0000858470-20-000019,2020,Q1,10-Q,2020-05-01,NaN
156,2020-01-01,2020-06-30,0.21,0000858470-21-000049,2021,Q2,10-Q,2021-07-30,NaN


In [68]:
# Example: allow any 4-digit year and require exact match
eps_table[
    (eps_table["frame"].str.match(r"^CY\d{4}Q[1-4]$", na=False))
    & (eps_table["fp"] == "Q3")
]

,start,end,val,accn,fy,fp,form,filed,frame
4,2008-07-01,2008-09-30,0.65,0001193125-09-217450,2009,Q3,10-Q,2009-10-29,CY2008Q3
12,2009-07-01,2009-09-30,0.38,0001193125-09-217450,2009,Q3,10-Q,2009-10-29,CY2009Q3
24,2010-07-01,2010-09-30,0.04,0001104659-11-058807,2011,Q3,10-Q,2011-10-28,CY2010Q3
37,2011-07-01,2011-09-30,0.14,0001104659-12-071549,2012,Q3,10-Q,2012-10-26,CY2011Q3
50,2012-07-01,2012-09-30,0.09,0001104659-13-077893,2013,Q3,10-Q,2013-10-25,CY2012Q3
63,2013-07-01,2013-09-30,0.17,0001445305-14-004439,2014,Q3,10-Q,2014-10-24,CY2013Q3
76,2014-07-01,2014-09-30,0.24,0000858470-15-000027,2015,Q3,10-Q,2015-10-23,CY2014Q3
89,2015-07-01,2015-09-30,-0.04,0000858470-16-000066,2016,Q3,10-Q,2016-10-28,CY2015Q3
103,2016-07-01,2016-09-30,-0.02,0000858470-17-000024,2017,Q3,10-Q,2017-10-30,CY2016Q3
117,2017-07-01,2017-09-30,0.04,0000858470-18-000056,2018,Q3,10-Q,2018-10-26,CY2017Q3


In [72]:
eps_table[(eps_table["start"].dt.month > 1) & (eps_table["fp"] == "FY")]

,start,end,val,accn,fy,fp,form,filed,frame
138,2019-04-01,2019-06-30,0.43,0000858470-21-000013,2020,FY,10-K,2021-02-26,CY2019Q2
147,2019-07-01,2019-09-30,0.22,0000858470-21-000013,2020,FY,10-K,2021-02-26,CY2019Q3
148,2019-10-01,2019-12-31,0.36,0000858470-21-000013,2020,FY,10-K,2021-02-26,CY2019Q4
158,2020-04-01,2020-06-30,0.08,0000858470-21-000013,2020,FY,10-K,2021-02-26,NaN
160,2020-07-01,2020-09-30,-0.04,0000858470-21-000013,2020,FY,10-K,2021-02-26,NaN
165,2020-10-01,2020-12-31,0.33,0000858470-21-000013,2020,FY,10-K,2021-02-26,CY2020Q4


In [31]:
filtered_eps[filtered_eps["fp"] == "FY"]

,start,end,eps,fp
1,2007-01-01,2007-12-31,1.73,FY
5,2008-01-01,2008-12-31,2.10,FY
10,2009-01-01,2009-12-31,0.72,FY
15,2010-01-01,2010-12-31,0.99,FY
20,2011-01-01,2011-12-31,0.59,FY
25,2012-01-01,2012-12-31,0.31,FY
30,2013-01-01,2013-12-31,0.67,FY
35,2014-01-01,2014-12-31,0.25,FY
40,2015-01-01,2015-12-31,-0.28,FY
45,2016-01-01,2016-12-31,-0.91,FY


In [7]:
equity_table = extract_quarterly_data(facts["facts"], 'StockholdersEquity', 'USD')

2025-11-11 09:58:32,587 - src.logger - INFO - Extracting quarterly data for StockholdersEquity in USD
2025-11-11 09:58:32,617 - src.logger - INFO - Successfully extracted quarterly data for StockholdersEquity


In [17]:
facts["facts"]["us-gaap"]["StockholdersEquity"]["units"]["USD"]

[{'end': '2006-12-31',
  'val': 945198000,
  'accn': '0001193125-10-042257',
  'fy': 2009,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2010-02-26',
  'frame': 'CY2006Q4I'},
 {'end': '2007-12-31',
  'val': 1070257000,
  'accn': '0001193125-10-042257',
  'fy': 2009,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2010-02-26'},
 {'end': '2007-12-31',
  'val': 1070257000,
  'accn': '0001193125-11-049690',
  'fy': 2010,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2011-02-28',
  'frame': 'CY2007Q4I'},
 {'end': '2008-12-31',
  'val': 1790562000,
  'accn': '0001193125-09-159490',
  'fy': 2009,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': '2009-07-30'},
 {'end': '2008-12-31',
  'val': 1790562000,
  'accn': '0001193125-09-217450',
  'fy': 2009,
  'fp': 'Q3',
  'form': '10-Q',
  'filed': '2009-10-29'},
 {'end': '2008-12-31',
  'val': 1790562000,
  'accn': '0001193125-10-042257',
  'fy': 2009,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2010-02-26'},
 {'end': '2008-12-31',
  'val': 1790562000,
  'accn':

In [19]:
equity_table.head(10)

,end,val,accn,fy,fp,form,filed,frame
0,2006-12-31,945198000,0001193125-10-042257,2009,FY,10-K,2010-02-26,CY2006Q4I
1,2007-12-31,1070257000,0001193125-10-042257,2009,FY,10-K,2010-02-26,NaN
2,2007-12-31,1070257000,0001193125-11-049690,2010,FY,10-K,2011-02-28,CY2007Q4I
3,2008-12-31,1790562000,0001193125-09-159490,2009,Q2,10-Q,2009-07-30,NaN
4,2008-12-31,1790562000,0001193125-09-217450,2009,Q3,10-Q,2009-10-29,NaN
5,2008-12-31,1790562000,0001193125-10-042257,2009,FY,10-K,2010-02-26,NaN
6,2008-12-31,1790562000,0001193125-11-049690,2010,FY,10-K,2011-02-28,NaN
7,2008-12-31,1790562000,0001047469-12-001751,2011,FY,10-K,2012-02-28,CY2008Q4I
8,2009-06-30,1836472000,0001193125-09-159490,2009,Q2,10-Q,2009-07-30,CY2009Q2I
9,2009-09-30,1823814000,0001193125-09-217450,2009,Q3,10-Q,2009-10-29,CY2009Q3I


In [24]:
equity_table[(equity_table["end"].dt.year == 2008) | (equity_table["frame"].str.startswith("CY2008", na=False))]

,end,val,accn,fy,fp,form,filed,frame
3,2008-12-31,1790562000,0001193125-09-159490,2009,Q2,10-Q,2009-07-30,NaN
4,2008-12-31,1790562000,0001193125-09-217450,2009,Q3,10-Q,2009-10-29,NaN
5,2008-12-31,1790562000,0001193125-10-042257,2009,FY,10-K,2010-02-26,NaN
6,2008-12-31,1790562000,0001193125-11-049690,2010,FY,10-K,2011-02-28,NaN
7,2008-12-31,1790562000,0001047469-12-001751,2011,FY,10-K,2012-02-28,CY2008Q4I


In [30]:
equity_table[equity_table["end"].dt.year == 2020].sort_values("filed")

,end,val,accn,fy,fp,form,filed,frame
149,2020-03-31,2168395000,0000858470-20-000019,2020,Q1,10-Q,2020-05-01,NaN
148,2020-03-31,2168395000,0000858470-20-000032,2020,Q2,10-Q,2020-07-31,NaN
151,2020-06-30,2165979000,0000858470-20-000032,2020,Q2,10-Q,2020-07-31,NaN
155,2020-09-30,2118488000,0000858470-20-000049,2020,Q3,10-Q,2020-10-30,NaN
150,2020-03-31,2168395000,0000858470-20-000049,2020,Q3,10-Q,2020-10-30,NaN
152,2020-06-30,2165979000,0000858470-20-000049,2020,Q3,10-Q,2020-10-30,NaN
165,2020-12-31,2215707000,0000858470-21-000013,2020,FY,10-K,2021-02-26,NaN
146,2020-03-31,2168395000,0000858470-21-000022,2021,Q1,10-Q,2021-04-30,NaN
164,2020-12-31,2215707000,0000858470-21-000022,2021,Q1,10-Q,2021-04-30,NaN
162,2020-12-31,2215707000,0000858470-21-000049,2021,Q2,10-Q,2021-07-30,NaN


In [ ]:
y_q_to_se_map = {}
for i, row in equity_table.iterrows():
    current_quarter = determine_quarter(row["end"])
    start, end = get_start_and_end_of_quarter(row["end"].year, current_quarter)
    row_dict = {
        "start": start,
        "end": end,
        "equity": row["val"],
        "fp": f"Q{current_quarter}"
    }